# Filmantic: MovieLens coverage check

We use MovieLens ratings as the answer key for evaluation. This notebook checks how well the TMDB catalog lines up with MovieLens.

In [1]:
import pandas as pd

movies = pd.read_csv("../data/raw/tmdb_5000_movies.csv")
ratings = pd.read_csv("../data/raw/ml-latest-small/ratings.csv")
links = pd.read_csv("../data/raw/ml-latest-small/links.csv")

In [2]:
print(movies.shape, ratings.shape, links.shape)

(4803, 20) (100836, 4) (9742, 3)


In [3]:
links.head(3)

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0


In [4]:
ratings.head(3)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224


In [5]:
links = links.dropna(subset=["tmdbId"])
links["tmdbId"] = links["tmdbId"].astype(int)

In [6]:
merged = ratings.merge(links[["movieId", "tmdbId"]], on="movieId")
print(merged.shape)

(100823, 5)


In [7]:
merged["in_catalog"] = merged["tmdbId"].isin(movies["id"])
print(merged["in_catalog"].mean())

0.6962101901351874


## Users we can evaluate

A rating of 4 or more counts as "liked". Evaluation hides one liked movie per user, so a user needs at least 5 liked movies in the catalog.

In [8]:
liked = merged[merged["in_catalog"] & (merged["rating"] >= 4)]
per_user = liked.groupby("userId").size()

print(len(per_user), (per_user >= 5).sum())

609 599


## Findings
- 70% of MovieLens ratings are for movies in our catalog
- 599 users have at least 5 liked movies, so we test on these users